In [2]:
import os
os.chdir(r"D:\Projects\Poverty Predictor Bd")
print(os.getcwd())

D:\Projects\Poverty Predictor Bd


In [4]:
import pandas as pd
import geopandas as gpd

districts = gpd.read_file("data/raw/shapefiles/bangladesh_districts_clean.gpkg")
ntl       = pd.read_csv("data/raw/ntl_2018_2022.csv")
pop       = pd.read_csv("data/raw/population_2020.csv")
ndvi      = pd.read_csv("data/raw/ndvi_2022.csv")
lc        = pd.read_csv("data/raw/landcover_2021.csv")
elev      = pd.read_csv("data/raw/elevation.csv")
road      = pd.read_csv("data/raw/road_network.csv")
poverty   = pd.read_csv("data/raw/poverty_labels_division.csv")

print("All files loaded!")
print(f"  districts: {len(districts)} | ntl: {ntl.shape} | pop: {pop.shape}")
print(f"  ndvi: {ndvi.shape} | lc: {lc.shape} | elev: {elev.shape}")
print(f"  road: {road.shape} | poverty: {poverty.shape}")

All files loaded!
  districts: 64 | ntl: (320, 9) | pop: (64, 4)
  ndvi: (64, 4) | lc: (64, 4) | elev: (64, 4)
  road: (64, 3) | poverty: (8, 14)


In [5]:
name_fix = {
    'Barisal':       'Barishal',
    'Chittagong':    'Chattogram',
    'Comilla':       'Cumilla',
    'Jessore':       'Jashore',
    'Bogra':         'Bogura',
    'Nawabganj':     'Chapai Nawabganj',
    'Brahamanbaria': 'Brahmanbaria',
    'Maulvibazar':   'Moulvibazar',
    'Netrakona':     'Netrokona',
    "Cox'S Bazar":   "Cox's Bazar",
}

def fix_names(df):
    df = df.copy()
    df['ADM2_NAME'] = df['ADM2_NAME'].replace(name_fix)
    df['ADM1_NAME'] = df['ADM1_NAME'].replace(name_fix)
    return df

ntl  = fix_names(ntl)
pop  = fix_names(pop)
ndvi = fix_names(ndvi)
lc   = fix_names(lc)
elev = fix_names(elev)

# Verify no old names remain
all_names = set(ntl['ADM2_NAME'].unique())
old_names = all_names & set(name_fix.keys())
print(f"Old names remaining: {old_names}")  # should be empty set()
print(f"Sample names: {sorted(list(all_names))[:8]}")

Old names remaining: set()
Sample names: ['Bagerhat', 'Bandarban', 'Barguna', 'Barishal', 'Bhola', 'Bogura', 'Brahmanbaria', 'Chandpur']


In [6]:
pop  = pop.rename(columns={'mean': 'pop_density', 'sum': 'pop_total'})
ndvi = ndvi.rename(columns={'mean': 'ndvi_mean', 'stdDev': 'ndvi_std'})
lc   = lc.rename(columns={'urban': 'urban_fraction', 'water': 'water_fraction'})
elev = elev.rename(columns={'mean': 'elevation_mean', 'stdDev': 'elevation_std'})
ntl  = ntl.rename(columns={
    'mean': 'ntl_mean', 'stdDev': 'ntl_std',
    'max':  'ntl_max',  'min':    'ntl_min',
    'p25':  'ntl_p25',  'p75':    'ntl_p75',
})

print("Columns renamed!")
print(f"  NTL cols:  {ntl.columns.tolist()}")
print(f"  Pop cols:  {pop.columns.tolist()}")
print(f"  NDVI cols: {ndvi.columns.tolist()}")

Columns renamed!
  NTL cols:  ['ADM2_NAME', 'ADM1_NAME', 'year', 'ntl_mean', 'ntl_std', 'ntl_max', 'ntl_min', 'ntl_p25', 'ntl_p75']
  Pop cols:  ['ADM2_NAME', 'ADM1_NAME', 'pop_density', 'pop_total']
  NDVI cols: ['ADM2_NAME', 'ADM1_NAME', 'ndvi_mean', 'ndvi_std']


In [7]:
# 2022 snapshot
ntl_2022 = ntl[ntl['year'] == 2022].drop(columns='year').copy()

# Temporal feature 1: trend slope (2018 -> 2022)
def compute_trend(group):
    years = group['year'].values
    means = group['ntl_mean'].values
    slope = (means[-1] - means[0]) / (years[-1] - years[0])
    return pd.Series({'ntl_trend': round(slope, 6)})

ntl_trend = ntl.groupby('ADM2_NAME').apply(
    compute_trend, include_groups=False
).reset_index()

# Temporal feature 2: year-on-year change 2021->2022
ntl_2021 = ntl[ntl['year'] == 2021][['ADM2_NAME', 'ntl_mean']].rename(
    columns={'ntl_mean': 'ntl_mean_2021'}
)
ntl_2022 = ntl_2022.merge(ntl_2021, on='ADM2_NAME')
ntl_2022['ntl_yoy_change'] = (ntl_2022['ntl_mean'] - ntl_2022['ntl_mean_2021']).round(6)
ntl_2022 = ntl_2022.drop(columns='ntl_mean_2021')

# Merge trend into ntl_2022
ntl_2022 = ntl_2022.merge(ntl_trend, on='ADM2_NAME')

print(f"NTL feature set: {ntl_2022.shape}")
print(f"Columns: {ntl_2022.columns.tolist()}")

NTL feature set: (64, 10)
Columns: ['ADM2_NAME', 'ADM1_NAME', 'ntl_mean', 'ntl_std', 'ntl_max', 'ntl_min', 'ntl_p25', 'ntl_p75', 'ntl_yoy_change', 'ntl_trend']


In [8]:
# Start from the clean shapefile as the base
master = districts[['district_name', 'division_name', 'geometry']].copy()

def merge_gee(base, df, feature_cols):
    """Merge a GEE dataframe into the master on district name."""
    cols = ['ADM2_NAME'] + feature_cols
    return base.merge(df[cols], left_on='district_name', 
                      right_on='ADM2_NAME', how='left').drop(columns='ADM2_NAME')

# Merge each dataset
master = merge_gee(master, ntl_2022, 
                   ['ntl_mean','ntl_std','ntl_max','ntl_min',
                    'ntl_p25','ntl_p75','ntl_yoy_change','ntl_trend'])
master = merge_gee(master, pop,  ['pop_density', 'pop_total'])
master = merge_gee(master, ndvi, ['ndvi_mean', 'ndvi_std'])
master = merge_gee(master, lc,   ['urban_fraction', 'water_fraction'])
master = merge_gee(master, elev, ['elevation_mean', 'elevation_std'])

# Road network (already has correct names)
master = master.merge(road[['district_name', 'road_length_km']], 
                      on='district_name', how='left')

# Poverty labels — division level join
master = master.merge(
    poverty[['division', 'hcr_upper_2022_national', 
             'hcr_lower_2022_national', 'hcr_upper_change_2016_2022']].rename(
        columns={'division': 'division_name',
                 'hcr_upper_2022_national': 'poverty_hcr',
                 'hcr_lower_2022_national': 'poverty_hcr_lower',
                 'hcr_upper_change_2016_2022': 'poverty_change'}
    ),
    on='division_name', how='left'
)

print(f"\nMaster shape: {master.shape}")
print(f"\nColumns:\n{master.columns.tolist()}")
print(f"\nNull counts:\n{master.isnull().sum()}")


Master shape: (64, 23)

Columns:
['district_name', 'division_name', 'geometry', 'ntl_mean', 'ntl_std', 'ntl_max', 'ntl_min', 'ntl_p25', 'ntl_p75', 'ntl_yoy_change', 'ntl_trend', 'pop_density', 'pop_total', 'ndvi_mean', 'ndvi_std', 'urban_fraction', 'water_fraction', 'elevation_mean', 'elevation_std', 'road_length_km', 'poverty_hcr', 'poverty_hcr_lower', 'poverty_change']

Null counts:
district_name        0
division_name        0
geometry             0
ntl_mean             0
ntl_std              0
ntl_max              0
ntl_min              0
ntl_p25              0
ntl_p75              0
ntl_yoy_change       0
ntl_trend            0
pop_density          0
pop_total            0
ndvi_mean            0
ndvi_std             0
urban_fraction       0
water_fraction       0
elevation_mean       0
elevation_std        0
road_length_km       0
poverty_hcr          0
poverty_hcr_lower    0
poverty_change       0
dtype: int64


In [9]:
# NTL per capita (normalizes for population size)
master['ntl_per_capita'] = master['ntl_mean'] / (master['pop_density'] + 1e-6)

# Road density (km per sq km of district area)
master['area_sqkm'] = master.geometry.to_crs('EPSG:32646').area / 1e6
master['road_density'] = master['road_length_km'] / master['area_sqkm']

# NTL interquartile range (spread of light within district)
master['ntl_iqr'] = master['ntl_p75'] - master['ntl_p25']

# Round all numeric columns to 6 decimal places
num_cols = master.select_dtypes(include='number').columns
master[num_cols] = master[num_cols].round(6)

print("Engineered features added!")
print(f"Final shape: {master.shape}")
print(f"\nNew features: ntl_per_capita, road_density, ntl_iqr, area_sqkm")
print(master[['district_name', 'ntl_per_capita', 
              'road_density', 'ntl_iqr']].head(8))

Engineered features added!
Final shape: (64, 27)

New features: ntl_per_capita, road_density, ntl_iqr, area_sqkm
  district_name  ntl_per_capita  road_density   ntl_iqr
0       Barguna        0.080818      1.732400  0.051739
1      Barishal        0.067637      1.167066  0.116509
2         Bhola        0.053214      1.364132  0.000000
3     Jhalokati        0.070316      0.753898  0.120625
4    Patuakhali        0.097026      1.139233  0.000000
5      Pirojpur        0.060558      2.411488  0.079217
6     Bandarban        0.384385      0.456217  0.055908
7  Brahmanbaria        0.063348      2.108506  0.373739


In [10]:
import os
os.makedirs("data/processed", exist_ok=True)

# Final null check
print("=== FINAL NULL CHECK ===")
nulls = master.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "No nulls!")

# Save CSV (for ML)
master.drop(columns='geometry').to_csv(
    "data/processed/master_features.csv", index=False
)

# Save GeoPackage (for mapping)
master.to_file(
    "data/processed/master_features.gpkg", driver="GPKG"
)

print(f"\nSaved master_features.csv  — {master.shape[0]} rows, {master.shape[1]-1} features")
print(f"Saved master_features.gpkg — includes geometry for mapping")
print(f"\nFinal dataset preview:")
print(master.drop(columns='geometry').head(5).to_string())

=== FINAL NULL CHECK ===
No nulls!

Saved master_features.csv  — 64 rows, 26 features
Saved master_features.gpkg — includes geometry for mapping

Final dataset preview:
  district_name division_name  ntl_mean   ntl_std    ntl_max  ntl_min   ntl_p25   ntl_p75  ntl_yoy_change  ntl_trend  pop_density     pop_total  ndvi_mean  ndvi_std  urban_fraction  water_fraction  elevation_mean  elevation_std  road_length_km  poverty_hcr  poverty_hcr_lower  poverty_change  ntl_per_capita    area_sqkm  road_density   ntl_iqr
0       Barguna      Barishal  0.448607  0.412899  14.070000    0.230  0.348863  0.400602        0.044712   0.035456     5.550811  7.803639e+05   0.553958  0.130788        0.002596        0.116690        5.678317       3.248722         2281.37         26.9               11.8             0.4        0.080818  1316.883927      1.732400  0.051739
1      Barishal      Barishal  0.556835  0.768384  13.570000    0.220  0.346415  0.462924        0.053557   0.033745     8.232663  1.969138e+